# 니케 캐릭터 · 수채화 LoRA 학습 (Kaggle T4 ×2)

SDXL(Animagine XL 4.0 Opt) 기반 LoRA를 학습합니다. T4 2장에 작업을 하나씩 맡겨 **동시에 2개**를 학습합니다.

## 사용 순서
1. **데이터셋 업로드**: zip을 Kaggle Datasets에 **Private**로 올립니다 (이름 예: `nikke-lora-data`).
   ```
   nikke-lora-data.zip
    ├ marian/            ← 기본 마리안 이미지
    ├ privaty_maid/      ← 프리바티: 언카인드 메이드
    ├ marciana_marine/   ← 마르차나: 마린 스터디
    └ style_watercolor/  ← 수채화 컷신
   ```
2. 이 노트북을 가져온 뒤(**File → Import Notebook**), 오른쪽 패널에서
   - **Add Input** → 올린 데이터셋 추가
   - **Settings → Accelerator: GPU T4 x2**, **Internet: On**
3. **1차: `MODE = "tag"`** → **Run All**
   - 자동 태그(캡션)를 만들고, 자주 나온 태그 표를 보여줍니다.
   - 오른쪽 **Output**의 `captions_*.zip`을 받아 확인·수정한 뒤, 데이터셋 폴더에 txt를 넣어 **새 버전**으로 올립니다.
4. **2차: `MODE = "train"`** → 오른쪽 위 **Save Version → Save & Run All**
   - 브라우저를 닫아도 학습이 계속됩니다.
   - 끝나면 **Output**에서 `samples_*.zip`(에폭별 샘플)을 보고 가장 좋은 에폭의 `.safetensors`를 받습니다.

> 게임 캡처 데이터와 결과 LoRA는 **개인 용도**로만 사용하고, 데이터셋은 반드시 Private로 두세요.

In [ ]:
# ===== ① 여기만 수정하세요 =====
MODE = "tag"   # "tag" = 자동 캡션 생성 / "train" = 학습

# 오른쪽 Input 패널에 보이는 데이터셋 경로
DATASET_ROOT = "/kaggle/input/nikke-lora-data"

BASE_MODEL_URL = "https://huggingface.co/cagliostrolab/animagine-xl-4.0/resolve/main/animagine-xl-4.0-opt.safetensors"

# 한 번에 최대 2개 (GPU 0, 1에 하나씩). folder는 데이터셋 안 폴더 이름과 같아야 합니다.
JOBS = [
    dict(folder="marian",           token="mrn_nikke",    kind="character", gpu=0),
    dict(folder="style_watercolor", token="nkwc_style",   kind="style",     gpu=1),
    # dict(folder="privaty_maid",    token="pvmaid_nikke", kind="character", gpu=0),
    # dict(folder="marciana_marine", token="mrcms_nikke",  kind="character", gpu=1),
]

# 캡션에서 지울 태그 (폴더별). tag 모드의 '자주 나온 태그' 표를 보고 채우세요.
#  - 캐릭터: 항상 같은 특징(머리색, 눈색, 기본 의상, 캐릭터 이름 태그) → 지우면 트리거 토큰에 묶임
#  - 수채화: 화풍 단어 → 지우면 nkwc_style 토큰에 묶임
REMOVE_TAGS = {
    "marian": [],
    "privaty_maid": [],
    "marciana_marine": [],
    "style_watercolor": ["watercolor (medium)", "traditional media", "painting (medium)"],
}

# 학습 설정 (보통은 그대로)
SETTINGS = {
    "character": dict(dim=32, alpha=16, lr=1e-4, epochs=10, steps_per_epoch=150),
    "style":     dict(dim=32, alpha=16, lr=1e-4, epochs=10, steps_per_epoch=200),
}
BATCH_SIZE = 2      # 메모리 부족(CUDA out of memory)이 나면 1로
RESOLUTION = 1024

## ② 환경 확인

In [ ]:
import os, sys, glob, shutil, subprocess, time, re, zipfile
from collections import Counter

def run(cmd, cwd=None):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, cwd=cwd)
    if r.returncode != 0:
        raise RuntimeError(f"명령 실패 (코드 {r.returncode}): {cmd}")

assert MODE in ("tag", "train"), 'MODE는 "tag" 또는 "train"이어야 합니다'
assert 1 <= len(JOBS) <= 2, "JOBS는 1~2개만 넣으세요 (GPU 2장)"
assert len({j["gpu"] for j in JOBS}) == len(JOBS), "작업마다 gpu 번호를 다르게 지정하세요 (0, 1)"
for j in JOBS:
    assert j["kind"] in SETTINGS, f'kind는 character 또는 style: {j}'

run("nvidia-smi --query-gpu=index,name,memory.total --format=csv")
print("Python", sys.version)
assert os.path.isdir(DATASET_ROOT), f"데이터셋 경로가 없습니다: {DATASET_ROOT}\n오른쪽 Input 패널에서 경로를 확인하세요."
print("데이터셋 내용:", os.listdir(DATASET_ROOT))

## ③ 학습 도구 설치 (sd-scripts)

In [ ]:
SD = "/tmp/sd-scripts"
if not os.path.isdir(SD):
    run(f"git clone --depth 1 --branch v0.12.3 https://github.com/kohya-ss/sd-scripts.git {SD}")
run("pip install -q -r requirements.txt", cwd=SD)
if MODE == "tag":
    run("pip install -q onnxruntime-gpu")

import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| GPU 수", torch.cuda.device_count())
assert torch.cuda.device_count() >= len(JOBS), "GPU가 부족합니다. Settings → Accelerator를 GPU T4 x2로 바꾸세요."

## ④ 데이터 복사 · 점검

In [ ]:
from PIL import Image

DATA = "/tmp/data"
IMG_EXT = (".png", ".jpg", ".jpeg", ".webp")
MIN_COUNT = {"character": 20, "style": 30}

def find_src(folder):
    # Kaggle가 zip을 풀 때 폴더가 한 단계 더 생기는 경우도 찾아봄
    for p in [os.path.join(DATASET_ROOT, folder)] + glob.glob(os.path.join(DATASET_ROOT, "*", folder)):
        if os.path.isdir(p):
            return p
    raise FileNotFoundError(f"데이터셋에서 '{folder}' 폴더를 찾지 못했습니다. 폴더 이름을 확인하세요.")

def images_of(d):
    return sorted(f for f in os.listdir(d) if f.lower().endswith(IMG_EXT))

for job in JOBS:
    src, dst = find_src(job["folder"]), os.path.join(DATA, job["folder"])
    if os.path.isdir(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    imgs = images_of(dst)
    n_cap = sum(os.path.exists(os.path.join(dst, os.path.splitext(f)[0] + ".txt")) for f in imgs)
    small = []
    for f in imgs:
        with Image.open(os.path.join(dst, f)) as im:
            if min(im.size) < 768:
                small.append(f"{f} {im.size}")
    job["n_images"] = len(imgs)
    job["has_captions"] = len(imgs) > 0 and n_cap == len(imgs)
    print(f"\n[{job['folder']}] 이미지 {len(imgs)}장, 캡션 {n_cap}개")
    if len(imgs) < MIN_COUNT[job["kind"]]:
        print(f"  ⚠ 권장 {MIN_COUNT[job['kind']]}장 이상 (품질을 위해)")
    if small:
        print(f"  ⚠ 짧은 변이 768px 미만인 이미지 {len(small)}장 (흐릿하게 학습될 수 있음):")
        for s in small[:20]:
            print("    -", s)

## ⑤ 캡션 도구

In [ ]:
def read_tags(path):
    with open(path, encoding="utf-8") as f:
        return [t.strip() for t in f.read().replace("\n", ",").split(",") if t.strip()]

def clean_captions(job):
    """지울 태그 제거 + 트리거 토큰을 맨 앞에 (여러 번 실행해도 안전)"""
    d = os.path.join(DATA, job["folder"])
    remove = {t.strip().lower() for t in REMOVE_TAGS.get(job["folder"], [])}
    for txt in glob.glob(os.path.join(d, "*.txt")):
        tags = [t for t in read_tags(txt) if t.lower() not in remove and t != job["token"]]
        with open(txt, "w", encoding="utf-8") as f:
            f.write(", ".join([job["token"]] + tags))

def tag_report(job, top=50):
    d = os.path.join(DATA, job["folder"])
    files = glob.glob(os.path.join(d, "*.txt"))
    c = Counter()
    for txt in files:
        c.update(set(read_tags(txt)[1:]))
    print(f"\n[{job['folder']}] 자주 나온 태그 (캡션 {len(files)}개 중)")
    for tag, n in c.most_common(top):
        print(f"  {n:4d}  {100 * n / max(1, len(files)):5.1f}%  {tag}")

## ⑥ [tag 모드] 자동 캡션 생성

- WD14 태거(`wd-eva02-large-tagger-v3`)로 태그를 만듭니다. **기존 txt는 덮어씁니다.**
- 표에서 **거의 모든 이미지에 나오는 고정 특징**(캐릭터의 머리색·의상, 수채화의 화풍 태그)을 `REMOVE_TAGS`에 넣으세요.
- 결과 `captions_<폴더>.zip`은 압축을 풀면 `<폴더>/파일.txt` 구조라서, 데이터셋 루트에 그대로 풀면 됩니다.

In [ ]:
if MODE == "tag":
    for job in JOBS:
        d = os.path.join(DATA, job["folder"])
        run(f'python finetune/tag_images_by_wd14_tagger.py --onnx '
            f'--repo_id SmilingWolf/wd-eva02-large-tagger-v3 --batch_size 4 --thresh 0.35 '
            f'--caption_extension .txt --remove_underscore "{d}"', cwd=SD)
        clean_captions(job)
        tag_report(job)
        zpath = f"/kaggle/working/captions_{job['folder']}.zip"
        with zipfile.ZipFile(zpath, "w") as z:
            for txt in sorted(glob.glob(os.path.join(d, "*.txt"))):
                z.write(txt, arcname=f"{job['folder']}/{os.path.basename(txt)}")
        print(f"→ 저장: {zpath}")
    print("\n✅ 캡션 생성 완료. REMOVE_TAGS를 채우고 캡션을 확인한 뒤, txt를 데이터셋에 넣어 새 버전으로 올리세요.")
else:
    print("train 모드: 이 단계는 건너뜁니다.")

## ⑦ [train 모드] 학습

In [ ]:
QUALITY = "masterpiece, high score, great score, absurdres"
NEG = ("lowres, bad anatomy, bad hands, text, error, missing fingers, extra digits, "
       "worst quality, low quality, watermark, signature")

def sample_prompts(job):
    t = job["token"]
    if job["kind"] == "character":
        bodies = ["1girl, solo, upper body, looking at viewer, smile, simple background",
                  "1girl, solo, full body, standing, outdoors, city street",
                  "1girl, solo, sitting, cafe, window, from side"]
    else:
        bodies = ["1girl, solo, upper body, looking at viewer",
                  "1girl, solo, standing, street, sky",
                  "no humans, scenery, ruins, sky"]
    return "\n".join(f"{t}, {b}, {QUALITY} --n {NEG} --w 832 --h 1216 --d {1001 + i} --l 5 --s 28"
                     for i, b in enumerate(bodies)) + "\n"

def write_configs(job, out):
    s = SETTINGS[job["kind"]]
    repeats = max(1, round(s["steps_per_epoch"] * BATCH_SIZE / job["n_images"]))
    job["repeats"] = repeats
    toml = f"""[[datasets]]
resolution = {RESOLUTION}
batch_size = {BATCH_SIZE}
enable_bucket = true
bucket_no_upscale = true
min_bucket_reso = 512
max_bucket_reso = 2048
bucket_reso_steps = 64

  [[datasets.subsets]]
  image_dir = "{os.path.join(DATA, job['folder'])}"
  num_repeats = {repeats}
  caption_extension = ".txt"
  shuffle_caption = true
  keep_tokens = 1
"""
    with open(os.path.join(out, "dataset.toml"), "w", encoding="utf-8") as f:
        f.write(toml)
    with open(os.path.join(out, "sample_prompts.txt"), "w", encoding="utf-8") as f:
        f.write(sample_prompts(job))
    total = job["n_images"] * repeats * s["epochs"] // BATCH_SIZE
    print(f"[{job['token']}] {job['n_images']}장 × 반복 {repeats} × {s['epochs']}에폭 ÷ 배치 {BATCH_SIZE} ≈ {total}스텝")

def train_cmd(job, out, model):
    s = SETTINGS[job["kind"]]
    return [sys.executable, "-m", "accelerate.commands.launch",
            "--num_processes", "1", "--num_machines", "1", "--mixed_precision", "fp16",
            "--dynamo_backend", "no", "--num_cpu_threads_per_process", "2",
            "sdxl_train_network.py",
            "--pretrained_model_name_or_path", model,
            "--dataset_config", os.path.join(out, "dataset.toml"),
            "--output_dir", out, "--output_name", job["token"], "--save_model_as", "safetensors",
            "--network_module", "networks.lora",
            "--network_dim", str(s["dim"]), "--network_alpha", str(s["alpha"]),
            "--network_train_unet_only",
            "--learning_rate", str(s["lr"]), "--unet_lr", str(s["lr"]),
            "--optimizer_type", "AdamW8bit", "--lr_scheduler", "cosine",
            "--max_train_epochs", str(s["epochs"]), "--save_every_n_epochs", "1",
            "--mixed_precision", "fp16", "--save_precision", "fp16", "--no_half_vae",
            "--gradient_checkpointing", "--sdpa",
            "--cache_latents", "--cache_latents_to_disk",
            "--min_snr_gamma", "5", "--seed", "42",
            "--max_data_loader_n_workers", "2", "--persistent_data_loader_workers",
            "--sample_prompts", os.path.join(out, "sample_prompts.txt"),
            "--sample_every_n_epochs", "1", "--sample_at_first", "--sample_sampler", "euler_a"]

def read_log(path):
    with open(path, encoding="utf-8", errors="ignore") as f:
        return f.read()

def last_progress(text):
    parts = [p for p in re.split(r"[\r\n]", text) if "steps:" in p]
    return parts[-1].strip()[:160] if parts else ""

if MODE == "train":
    for job in JOBS:
        assert job["has_captions"], (f"[{job['folder']}] 캡션(txt)이 없거나 일부 빠졌습니다. "
                                     "먼저 tag 모드로 캡션을 만들어 데이터셋에 넣으세요.")
        clean_captions(job)

    MODEL = "/tmp/models/" + BASE_MODEL_URL.split("/")[-1].split("?")[0]
    if not (os.path.exists(MODEL) and os.path.getsize(MODEL) > 1e9):
        os.makedirs("/tmp/models", exist_ok=True)
        run(f'wget -nv -O "{MODEL}" "{BASE_MODEL_URL}"')
    print("베이스 모델:", MODEL, f"{os.path.getsize(MODEL) / 1e9:.2f}GB")

    procs = {}
    for i, job in enumerate(JOBS):
        out = f"/kaggle/working/output/{job['token']}"
        os.makedirs(out, exist_ok=True)
        write_configs(job, out)
        log_path = os.path.join(out, "train.log")
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(job["gpu"]))
        p = subprocess.Popen(train_cmd(job, out, MODEL), cwd=SD, env=env,
                             stdout=open(log_path, "w"), stderr=subprocess.STDOUT)
        procs[job["token"]] = dict(p=p, log=log_path, started=False, done=False)
        print(f"▶ [{job['token']}] 준비 중 (GPU {job['gpu']}) — 모델 로딩·캐시 생성")
        # 다음 작업은 이 작업이 실제 학습을 시작한 뒤 띄움 (모델 로딩 때 RAM이 겹치지 않게)
        if i < len(JOBS) - 1:
            while p.poll() is None and "running training" not in read_log(log_path):
                time.sleep(20)

    # 진행 상황 모니터링 (이 셀이 끝날 때까지 세션 유지)
    while not all(v["done"] for v in procs.values()):
        for name, v in procs.items():
            if v["done"]:
                continue
            text = read_log(v["log"])
            if not v["started"] and "running training" in text:
                v["started"] = True
                print(f"✅ [{name}] 학습 시작!", flush=True)
            code_ = v["p"].poll()
            if code_ is not None:
                v["done"] = True
                if code_ == 0:
                    print(f"🎉 [{name}] 학습 완료", flush=True)
                else:
                    print(f"❌ [{name}] 실패 (코드 {code_}). 로그 마지막 부분:", flush=True)
                    print("\n".join(re.split(r"[\r\n]", text)[-40:]))
            elif v["started"]:
                print(f"[{name}] {last_progress(text)}", flush=True)
        time.sleep(60)
else:
    print("tag 모드: 학습은 건너뜁니다.")

## ⑧ 결과 정리

In [ ]:
if MODE == "train":
    for job in JOBS:
        out = f"/kaggle/working/output/{job['token']}"
        loras = sorted(glob.glob(os.path.join(out, "*.safetensors")))
        print(f"\n[{job['token']}] LoRA {len(loras)}개")
        for f in loras:
            print(f"  {os.path.basename(f)}  {os.path.getsize(f) / 1e6:.0f}MB")
        sample_dir = os.path.join(out, "sample")
        if os.path.isdir(sample_dir):
            zpath = f"/kaggle/working/samples_{job['token']}.zip"
            with zipfile.ZipFile(zpath, "w") as z:
                for f in sorted(glob.glob(os.path.join(sample_dir, "*"))):
                    z.write(f, arcname=os.path.basename(f))
            print(f"  샘플 이미지 → {zpath}")
    print("\n샘플을 에폭별로 비교해서 가장 좋은 에폭의 .safetensors만 받으세요 "
          "(파일명의 -000005 = 5에폭, 번호 없는 파일 = 마지막 에폭).")